In [6]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
llm = init_chat_model("google_genai:gemini-2.0-flash")

In [25]:
llm.invoke("what is the capital of india ?").content

'The capital of India is **New Delhi**.'

In [29]:
class chatstate(TypedDict):
    messages : Annotated[list[BaseMessage], add_messages]

In [30]:
def chatbot(state:chatstate):
    msg = state['messages']
    response = llm.invoke(msg).content
    return {"messages": [response]}

In [31]:
checkpointer = InMemorySaver()

build = StateGraph(chatstate)

build.add_node("chatbot", chatbot)

build.add_edge(START, "chatbot")
build.add_edge("chatbot", END)


In [32]:
chatbot = build.compile(checkpointer=checkpointer)

In [37]:
thread_id = "1"

while True:
    user_message = input("Type Here: ")

    print("User:", user_message)

    if user_message.strip().lower() in ["exit", "bye"]:
        break

    config = {"configurable": {"thread_id": thread_id}}

    # Correct way: pass list of messages
    response = chatbot.invoke({"messages": [HumanMessage(content=user_message)]}, config=config)

    print("AI:", response["messages"][-1].content)

User: who are you ?
AI: I am a large language model, trained by Google.
User: what is the capital of india ?
AI: The capital of India is **New Delhi**.
User: my name is tark
AI: Okay, nice to meet you, Tark. How can I help you today?
User: i have skills in AI/ML feild
AI: That's fantastic, Tark! The AI/ML field is incredibly dynamic and in high demand. What are some of your specific skills within AI/ML? Knowing that will help me understand what you're good at and how I might be able to help you further. For example, do you have experience with:

*   **Specific Programming Languages:** Python, R, etc.
*   **Machine Learning Algorithms:** Regression, Classification, Clustering, Deep Learning (CNNs, RNNs, Transformers), etc.
*   **Deep Learning Frameworks:** TensorFlow, PyTorch, Keras, etc.
*   **Data Analysis and Visualization:** Pandas, NumPy, Matplotlib, Seaborn, etc.
*   **Cloud Platforms:** AWS, Azure, GCP, etc.
*   **Specific AI Applications:** Natural Language Processing (NLP), Com

In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, AIMessage
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

# Load model
llm = init_chat_model("google_genai:gemini-2.0-flash")

# Test basic call
print(llm.invoke("who are you?").content)

# Define State
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

# Node function
def chatbot_node(state: ChatState):
    msgs = state["messages"]
    response = llm.invoke(msgs).content
    return {"messages": [AIMessage(content=response)]}

# Checkpointing
checkpointer = InMemorySaver()

# Build graph
builder = StateGraph(ChatState)
builder.add_node("chatbot", chatbot_node)

builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

chatbot = builder.compile(checkpointer=checkpointer)


I am a large language model, trained by Google.


In [8]:
thread_id = "1"
config = {"configurable": {"thread_id": thread_id}}

In [9]:
result = chatbot.invoke({"messages": [HumanMessage(content="what is the capital of india ?")]}, config=config)
print(result["messages"][-1].content)

The capital of India is **New Delhi**.
